# Phase 2: Data Cleaning & Master Dataset

This notebook pulls all three raw tables from MySQL, cleans and merges them into a single master dataset, and writes it back to MySQL.

**Steps:**
1. Load `game_prices`, `cpi_data`, and `taketwo_financials` from MySQL
2. Inflation-adjust all prices to 2025 dollar terms using CPI data
3. Merge datasets by year into one master DataFrame
4. Run data quality checks
5. Write the master dataset back to MySQL

**Why inflation-adjust?**
A game priced at $59.99 in 2007 is not the same as $59.99 in 2025. Adjusting all prices to a common base year lets the regression model compare real purchasing power across time rather than nominal sticker prices. Without this step the model would see no meaningful price trend across 16 years.

**Output:** Populated `master_dataset` table in MySQL

## 1. Imports and Database Connection

In [ ]:
import pandas as pd
import numpy as np
import mysql.connector
import os
import sys
from dotenv import load_dotenv
from urllib.parse import quote_plus
from sqlalchemy import create_engine

sys.path.append(os.path.abspath('..'))
load_dotenv(dotenv_path='../.env')

print('Imports successful')

In [ ]:
def get_connection():
    """Raw MySQL connection for write operations."""
    return mysql.connector.connect(
        host=os.getenv('DB_HOST'),
        user=os.getenv('DB_USER'),
        password=os.getenv('DB_PASSWORD'),
        database=os.getenv('DB_NAME')
    )

def get_engine():
    """SQLAlchemy engine for pandas read operations."""
    password = quote_plus(os.getenv('DB_PASSWORD'))
    return create_engine(
        f"mysql+mysqlconnector://{os.getenv('DB_USER')}:{password}"
        f"@{os.getenv('DB_HOST')}/{os.getenv('DB_NAME')}"
    )

try:
    conn = get_connection()
    print(f"Connected to: {os.getenv('DB_NAME')}")
    conn.close()
except Exception as e:
    print(f"Connection failed: {e}")

## 2. Load Raw Tables from MySQL

In [ ]:
engine = get_engine()

df_prices = pd.read_sql("SELECT * FROM game_prices ORDER BY release_year", engine)
df_cpi = pd.read_sql("SELECT year, annual_cpi FROM cpi_data ORDER BY year", engine)
df_financials = pd.read_sql("""
    SELECT fiscal_year, total_revenue_usd_millions,
           gross_margin_pct, net_income_usd_millions
    FROM taketwo_financials
    ORDER BY fiscal_year
""", engine)

print(f"game_prices:       {len(df_prices)} records")
print(f"cpi_data:          {len(df_cpi)} records")
print(f"taketwo_financials:{len(df_financials)} records")

## 3. Preview Raw Data

In [ ]:
print("Game prices sample:")
df_prices.head(10)

In [ ]:
print("CPI data sample:")
df_cpi.tail(10)

In [ ]:
print("Take-Two financials:")
df_financials

## 4. Inflation Adjustment

Convert all nominal prices to 2025 dollar terms.

**Formula:** `real_price = nominal_price × (CPI_2025 / CPI_release_year)`

**Example:** COD4 launched at $59.99 in 2007. CPI in 2007 was ~207. CPI in 2025 is ~322.
Real price = $59.99 × (322 / 207) = ~$93.30 in today's money.

This means Call of Duty 4 was actually a more expensive game in real terms than any title released at $69.99 in 2023 — a genuine and counterintuitive insight your dashboard will visualise.

In [ ]:
# Get 2025 as the base year CPI
cpi_2025 = df_cpi.loc[df_cpi['year'] == 2025, 'annual_cpi'].values[0]
print(f"Base year CPI (2025): {cpi_2025}")

# Merge prices with CPI on release year
df = df_prices.merge(
    df_cpi.rename(columns={'year': 'release_year', 'annual_cpi': 'annual_cpi'}),
    on='release_year',
    how='left'
)

# Check for any years missing CPI data
missing = df[df['annual_cpi'].isna()]['release_year'].unique()
if len(missing) > 0:
    print(f"Warning: Missing CPI data for years: {missing}")
else:
    print("CPI data found for all release years")

# Calculate inflation multiplier and real prices
df['cpi_2025'] = cpi_2025
df['inflation_multiplier'] = (cpi_2025 / df['annual_cpi']).round(4)
df['base_price_real'] = (df['base_price_usd'] * df['inflation_multiplier']).round(2)
df['premium_price_real'] = (df['premium_price_usd'] * df['inflation_multiplier']).round(2)

print(f"\nInflation adjustment complete")
df[['game_title', 'release_year', 'base_price_usd', 'inflation_multiplier', 'base_price_real']].head(10)

## 5. Key Insight — Real Price Trend

Before merging, let's look at what the inflation adjustment reveals. This is one of the core insights of the project.

In [ ]:
# Show real vs nominal prices sorted by year
# This reveals the counterintuitive truth: older games were more expensive in real terms
insight = df[['game_title', 'publisher', 'release_year', 'base_price_usd', 'base_price_real', 'inflation_multiplier']].copy()
insight = insight.sort_values('release_year')
insight.columns = ['Game', 'Publisher', 'Year', 'Nominal Price', 'Real Price (2025 $)', 'Inflation Multiplier']

print("Real vs Nominal Prices — what games actually cost in today's money:")
print()
insight.to_string(index=False)

In [ ]:
# Average real price by year — shows the true price trend
avg_by_year = df.groupby('release_year')['base_price_real'].mean().round(2)
print("Average real base price by year (2025 dollars):")
print(avg_by_year.to_string())
print(f"\nOverall average real price: ${df['base_price_real'].mean():.2f}")
print(f"Overall average nominal price: ${df['base_price_usd'].mean():.2f}")

## 6. Merge with Take-Two Financials

We add Take-Two financial context to the dataset. This will be used for dashboard visualisation — showing how Take-Two's revenue growth aligns with the industry pricing shift — rather than as a model feature.

Note: Take-Two's fiscal year ends March 31, so their reported year lags calendar year by one quarter. We match on calendar year directly and note this offset in the README.

In [ ]:
df_master = df.merge(
    df_financials.rename(columns={'fiscal_year': 'release_year'}),
    on='release_year',
    how='left'
)

print(f"Master dataset shape: {df_master.shape}")
print(f"\nColumns: {list(df_master.columns)}")

# Check how many rows have Take-Two financial data
with_financials = df_master['total_revenue_usd_millions'].notna().sum()
print(f"\nRows with Take-Two financial data: {with_financials} of {len(df_master)}")
print("Note: Only Rockstar titles will have matching Take-Two financials by year")

## 7. Data Quality Checks

Before writing to MySQL, run systematic checks to catch any issues.

In [ ]:
print("=== DATA QUALITY REPORT ===")
print()

# Shape
print(f"Total records: {len(df_master)}")
print(f"Total columns: {len(df_master.columns)}")
print()

# Null counts for key columns
key_cols = ['game_title', 'publisher', 'release_year', 'platform_generation',
            'base_price_usd', 'base_price_real', 'inflation_multiplier']
print("Null counts (key columns):")
print(df_master[key_cols].isnull().sum().to_string())
print()

# Price range sanity
print(f"Base price range: ${df_master['base_price_usd'].min()} to ${df_master['base_price_usd'].max()}")
print(f"Real price range: ${df_master['base_price_real'].min()} to ${df_master['base_price_real'].max()}")
print()

# Year range
print(f"Year range: {df_master['release_year'].min()} to {df_master['release_year'].max()}")
print()

# Publisher distribution
print("Records per publisher:")
print(df_master['publisher'].value_counts().to_string())
print()

# Platform generation distribution
print("Records per platform generation:")
print(df_master['platform_generation'].value_counts().sort_index().to_string())
print()

# Duplicate check
dupes = df_master[df_master.duplicated(subset=['game_title', 'release_year'], keep=False)]
print(f"Duplicates: {len(dupes)} records")

## 8. Write Master Dataset to MySQL

In [ ]:
def write_master_dataset(df):
    """
    Write the cleaned master dataset to MySQL.
    Clears existing data first so this is safe to rerun.
    """
    conn = get_connection()
    cursor = conn.cursor()

    cursor.execute("DELETE FROM master_dataset")

    insert_query = """
        INSERT INTO master_dataset (
            game_title, publisher, release_year, platform,
            platform_generation, had_premium_edition,
            base_price_nominal, premium_price_nominal,
            base_price_real, premium_price_real,
            annual_cpi, cpi_2025, inflation_multiplier,
            revenue_usd_millions, gross_margin_pct, net_income_usd_millions
        ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    """

    def val(v):
        """Convert NaN to None for MySQL compatibility."""
        return None if pd.isna(v) else v

    records = []
    for _, row in df.iterrows():
        records.append((
            row['game_title'],
            row['publisher'],
            int(row['release_year']),
            row['platform'],
            int(row['platform_generation']),
            int(row['had_premium_edition']),
            val(row['base_price_usd']),
            val(row['premium_price_usd']),
            val(row['base_price_real']),
            val(row['premium_price_real']),
            val(row['annual_cpi']),
            val(row['cpi_2025']),
            val(row['inflation_multiplier']),
            val(row.get('total_revenue_usd_millions')),
            val(row.get('gross_margin_pct')),
            val(row.get('net_income_usd_millions')),
        ))

    cursor.executemany(insert_query, records)
    conn.commit()
    print(f"Written {cursor.rowcount} records to master_dataset")
    cursor.close()
    conn.close()

write_master_dataset(df_master)

## 9. Verify from MySQL

Read back from MySQL to confirm the round-trip was clean.

In [ ]:
df_verify = pd.read_sql("""
    SELECT
        release_year,
        publisher,
        game_title,
        platform_generation,
        base_price_nominal,
        base_price_real,
        inflation_multiplier
    FROM master_dataset
    ORDER BY release_year, publisher
""", engine)

print(f"Records in master_dataset: {len(df_verify)}")
df_verify

In [ ]:
print("Phase 2 complete. Master dataset ready in MySQL.")
print("Next step: notebooks/03_model.ipynb")